# Module 1: First model in an hour

**Project:** train an image classifier on Fashion-MNIST (28x28 grayscale clothing images, 10 classes).

**Done when:** the model trains, the loss goes down each epoch, and test accuracy is 85% or higher.

**How to use this notebook**
1. Turn on a GPU: *Runtime > Change runtime type > T4 GPU*.
2. Save your own copy first: *File > Save a copy in Drive*.
3. Run cells from top to bottom (Shift+Enter).
4. Fill in every cell marked `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
5. Stuck? Use the hint ladder: docs, then the hints below each exercise, then ask in the course forum.

## Step 1: Check your hardware

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("Tip: turn on a GPU via Runtime > Change runtime type. Training still works on CPU, just slower.")

## Step 2: Load and look at the data

This cell is given. Run it, then answer the question in the next cell.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

tfm = transforms.ToTensor()
train_ds = datasets.FashionMNIST("data", train=True, download=True, transform=tfm)
test_ds = datasets.FashionMNIST("data", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

x, y = next(iter(train_dl))
print("Batch of images:", x.shape)
print("Batch of labels:", y.shape, y[:10])

**Question:** what do the numbers in `torch.Size([64, 1, 28, 28])` mean? Write your answer here (double-click to edit):

*Your answer:*

**Exercise 2 (visual):** show 8 images from `x` with their class names as titles.

*Hint:* `import matplotlib.pyplot as plt`, then `plt.imshow(image.squeeze(), cmap="gray")` for each image.

In [ ]:
import matplotlib.pyplot as plt

# YOUR CODE HERE: show 8 images from x with their labels (use CLASSES[y[i]] as the title)


## Step 3: Build the model

Add three layers after `nn.Flatten()`:
1. A Linear layer from 28*28 inputs to 128 outputs
2. A ReLU activation
3. A Linear layer from 128 to 10 outputs (one per class)

*Hint:* `nn.Linear(in_features, out_features)` and `nn.ReLU()`.

In [ ]:
from torch import nn

model = nn.Sequential(
    nn.Flatten(),
    # YOUR CODE HERE: Linear(28*28 -> 128), ReLU, Linear(128 -> 10)
).to(device)

### Self-check: model

In [ ]:
out = model(torch.randn(4, 1, 28, 28).to(device))
assert out.shape == (4, 10), f"Expected output shape (4, 10) but got {tuple(out.shape)}. Did you add all three layers?"
print("Model check passed. Output shape:", tuple(out.shape))

## Step 4: Loss and optimizer

Create an **Adam** optimizer for the model's parameters with a learning rate of `1e-3`.

*Hint:* `torch.optim.Adam(model.parameters(), lr=1e-3)`

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = None  # YOUR CODE HERE: create the Adam optimizer

## Step 5: The training loop

Inside the loop, write the five steps **in this order**:
1. Forward pass: get predictions from `model(x)`
2. Compute the loss with `loss_fn(predictions, y)` and store it in a variable named `loss`
3. Zero the gradients: `optimizer.zero_grad()`
4. Backward pass: `loss.backward()`
5. Optimizer step: `optimizer.step()`

In [ ]:
loss = None
history = []   # last-batch loss of each epoch

for epoch in range(5):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        # YOUR CODE HERE: the five steps above

    if loss is not None:
        history.append(loss.item())
        print(f"epoch {epoch + 1}: last batch loss {loss.item():.3f}")
    else:
        print("loss is still None. Fill in the training steps above.")
        break

### Self-check: training

In [ ]:
assert len(history) == 5, "Training did not complete 5 epochs. Check your loop code."
assert history[-1] < history[0], "The loss did not go down. Check the order of the five steps."
print("Training check passed. Loss went from", round(history[0], 3), "to", round(history[-1], 3))

## Step 6: Evaluate honestly

Finish `evaluate(dl)`: loop over `dl` with gradient tracking turned off (`torch.no_grad()`), count how many predictions match the labels, and return accuracy.

*Hint:* predictions are `model(x).argmax(1)`; correct ones are `(preds == y).sum().item()`.

In [ ]:
def evaluate(dl):
    model.eval()
    correct, total = 0, 0
    # YOUR CODE HERE: loop over dl, use torch.no_grad(), update correct and total
    return correct / total if total else 0.0

test_acc = evaluate(test_dl)
print(f"Test accuracy: {test_acc:.3f}")

### Self-check: accuracy

In [ ]:
assert test_acc > 0.85, f"Test accuracy is {test_acc:.3f}, below the 85% target. Check evaluate() and your training loop."
print("Accuracy check passed. You are done with the core exercise!")

## Step 7: Look at your mistakes

Show 5 test images the model got **wrong**, with the true and predicted class names in the title.

In [ ]:
model.eval()
# YOUR CODE HERE: find 5 wrong predictions in the test set and plot them


## Stretch challenge (optional)

Change one thing and try to beat your own accuracy: use CIFAR-10 instead, add a second hidden layer, or try 10 epochs. Write down what you changed and what happened:

*Your notes:*

## Save your model (needed for the live demo)

This saves the trained weights. Download `model.pt` and use it in the Hugging Face demo.

In [ ]:
torch.save(model.state_dict(), "model.pt")
try:
    from google.colab import files
    files.download("model.pt")
except ImportError:
    print("Saved model.pt in the current folder.")

## Reflection and submission

Write three lines in your portfolio repo's `module-01/README.md`:
1. What you built
2. What broke
3. What you would change next time

Then download this notebook (*File > Download > .ipynb*), push it to `module-01` in your portfolio repo, and submit the link in Moodle.